In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle   
import pandas as pd
import numpy as np

In [2]:
#loading the model
model=load_model('churn_model.h5')

#load encoders and scaler
with open('label_encoder.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('onehot_encoder.pkl', 'rb') as f:
    onehot_encoder = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [4]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [6]:
geo_encoded = onehot_encoder.transform(
    [[input_data['Geography']]]
)
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder.get_feature_names_out(['Geography'])) 
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:
#combine ohe data with input data
input_data= pd.concat([pd.DataFrame(input_data, index=[0]), geo_encoded_df], axis=1)
input_data.drop('Geography', axis=1, inplace=True)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
#encodde categorical data
input_data['Gender'] = label_encoder_gender.transform(input_data['Gender'])
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
#scale the data
input_data_scaled = scaler.transform(input_data)
input_data_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [11]:
#predict the output
prediction = model.predict(input_data_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step


array([[0.035578]], dtype=float32)

In [14]:
prediction_probability = prediction[0][0]
prediction_probability

np.float32(0.035578)

In [15]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")       
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
